# CogVideoX-5B — Colab T4

**Универсальная генерация видео** с CogVideoX-5B (THUDM/ZhipuAI) на бесплатном T4 GPU.

```
[1. GPU] --> [2. Установка] --> [3. Модели] --> [4. Воркфлоу] --> [5. Медиа] --> [6. Запуск]
   |              |                 |               |               |              |
   v              v                 v               v               v              v
 Проверка     ComfyUI +         ~10 ГБ          JSON из        Фото для       Cloudflare
  CUDA       кастомные ноды    INT8+FP8+VAE      gist         I2V режима       туннель
```

---

### Когда выбирать этот ноутбук?

CogVideoX-5B — лучший выбор, когда вам нужно:

- **Стилизованный контент** — плавная, кинематографичная эстетика, отличная от реалистичного Wan 2.2
- **Быстрые итерации** — модель компактнее (~10 ГБ vs ~28 ГБ у Wan), скачивается и генерирует быстрее
- **Разнообразие в пайплайне** — другой «почерк» генерации для разнообразия вашего контента
- **Режим I2V** — анимация статичных изображений с сохранением стиля

> Если вам нужен максимальный реализм и детализация — используйте `colab_wan_video.ipynb`.
> Если нужна скорость при T2V — попробуйте `colab_hunyuan_video.ipynb`.

---

### Сравнение моделей
| | CogVideoX-5B | Wan 2.2 14B | HunyuanVideo 1.5 |
|---|---|---|---|
| **Размер модели** | ~5 ГБ (INT8) | ~14 ГБ (FP8) | ~7.76 ГБ (FP8) |
| **Общий размер скачивания** | ~10 ГБ | ~28 ГБ | ~15 ГБ |
| **Режимы** | T2V, I2V, Video extend | T2V, I2V, V2V, Talking | T2V |
| **Стиль** | Плавный, кинематографичный | Реалистичный | Реалистичный |
| **Скорость генерации** | Быстрая | Медленная | Средняя |
| **Разрешение** | 720x480 | до 832x480 | 848x480 |
| **Лучше всего для** | Стилизованный контент, разнообразие | Реализм, сложные сцены | Быстрые итерации T2V |

---

**Запускайте ячейки 1-6 по порядку**, затем откройте ссылку на туннель.

In [ ]:
#@title 1. Проверка GPU
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
import torch
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} ГБ")

In [ ]:
#@title 2. Установка ComfyUI + Кастомные ноды
%cd /content
!test -d ComfyUI || git clone https://github.com/comfyanonymous/ComfyUI.git
%cd /content/ComfyUI

# Фиксируем numpy < 2.0
!echo "numpy<2.0" > /tmp/numpy_constraint.txt
!pip install "numpy>=1.26,<2.0" -q
!pip install -r requirements.txt -q -c /tmp/numpy_constraint.txt

# Кастомные ноды
%cd /content/ComfyUI/custom_nodes
!test -d ComfyUI-CogVideoXWrapper || git clone https://github.com/kijai/ComfyUI-CogVideoXWrapper.git
!test -d ComfyUI-VideoHelperSuite || git clone https://github.com/Kosinkadink/ComfyUI-VideoHelperSuite.git
!test -d ComfyUI-KJNodes || git clone https://github.com/kijai/ComfyUI-KJNodes.git

!pip install -r ComfyUI-CogVideoXWrapper/requirements.txt -q -c /tmp/numpy_constraint.txt
!pip install -r ComfyUI-VideoHelperSuite/requirements.txt -q -c /tmp/numpy_constraint.txt
!pip install -r ComfyUI-KJNodes/requirements.txt -q -c /tmp/numpy_constraint.txt

print("\nУстановлено!")

In [ ]:
#@title 3. Скачивание моделей CogVideoX (~10 ГБ)
import os

M = "/content/ComfyUI/models"
os.makedirs(f"{M}/diffusion_models", exist_ok=True)
os.makedirs(f"{M}/text_encoders", exist_ok=True)
os.makedirs(f"{M}/vae", exist_ok=True)
os.makedirs(f"{M}/clip", exist_ok=True)

HF = "https://huggingface.co/Kijai/CogVideoX_comfy/resolve/main"

files = {
    # CogVideoX-5B transformer INT8 (~5 ГБ)
    f"{M}/diffusion_models/cogvideox_5b_I2V_fp8_e4m3fn.safetensors":
        f"{HF}/cogvideox_5b_I2V_fp8_e4m3fn.safetensors",
    # T5 XXL текстовый энкодер FP8 (~4.9 ГБ)
    f"{M}/text_encoders/t5xxl_fp8_e4m3fn.safetensors":
        "https://huggingface.co/mcmonkey/google_t5-v1_1-xxl_encoderonly/resolve/main/t5xxl_fp8_e4m3fn.safetensors",
    # CogVideoX VAE (~400 МБ)
    f"{M}/vae/cogvideox_vae.safetensors":
        f"{HF}/cogvideox_vae.safetensors",
}

for dest, url in files.items():
    if os.path.exists(dest):
        print(f"Уже существует: {os.path.basename(dest)}")
    else:
        print(f"\nСкачивание {os.path.basename(dest)}...")
        !wget -q --show-progress -O "{dest}" "{url}"

# Валидация скачивания
for dest in files:
    if os.path.exists(dest) and os.path.getsize(dest) < 1024:
        os.remove(dest)
        print(f"  ОШИБКА: {os.path.basename(dest)} не скачан — перезапустите ячейку")

print("\nВсе модели готовы!")
!du -sh {M}/diffusion_models/* {M}/text_encoders/* {M}/vae/*

In [ ]:
#@title 4. Скачивание воркфлоу
import os

GIST = "ea1ab4a53e1be1b8401e5031bcdae4f3"
RAW = f"https://gist.githubusercontent.com/russiankendricklamar/{GIST}/raw"
WF_DIR = "/content/ComfyUI/user/default/workflows"
os.makedirs(WF_DIR, exist_ok=True)

!wget -q -O "{WF_DIR}/video_cogvideo_t2v.json" "{RAW}/video_cogvideo_t2v.json"
print("Воркфлоу скачан!")

In [ ]:
#@title 5. Загрузка медиа (фото для режима I2V)
#@markdown ### Эта ячейка — для режима Image-to-Video (опциональная)
#@markdown
#@markdown Если вы используете **T2V** (текст-в-видео) — **пропустите эту ячейку** и переходите к шагу 6.
#@markdown
#@markdown Если вы хотите **анимировать изображение** (I2V):
#@markdown 1. Загрузите фото ниже
#@markdown 2. Поддерживаемые форматы: `.png`, `.jpg`, `.jpeg`, `.webp`
#@markdown 3. Рекомендуемое разрешение: **720x480** или близкое (16:9 / 3:2)
#@markdown 4. Файл появится в ComfyUI в ноде `LoadImage`

from google.colab import files
from IPython.display import display, Image as IPImage
import os

INPUT_DIR = "/content/ComfyUI/input"
os.makedirs(INPUT_DIR, exist_ok=True)

SUPPORTED_FORMATS = {".png", ".jpg", ".jpeg", ".webp"}

print("=" * 60)
print("  ЗАГРУЗКА МЕДИА ДЛЯ РЕЖИМА I2V")
print("=" * 60)
print()
print("Поддерживаемые форматы: PNG, JPG, JPEG, WEBP")
print("Рекомендуемое разрешение: 720x480 (или близкое)")
print()

uploaded = files.upload()

if not uploaded:
    print("\nФайлы не загружены. Если вам нужен T2V — просто перейдите к шагу 6.")
else:
    accepted = []
    rejected = []

    for name, data in uploaded.items():
        ext = os.path.splitext(name)[1].lower()
        if ext not in SUPPORTED_FORMATS:
            rejected.append((name, ext))
            continue
        path = os.path.join(INPUT_DIR, name)
        with open(path, "wb") as f:
            f.write(data)
        accepted.append(name)

    # Отчёт о загрузке
    print()
    print("-" * 60)

    if rejected:
        print("\n  ОТКЛОНЕНО (неподдерживаемый формат):")
        for name, ext in rejected:
            print(f"    {name} ({ext}) — используйте PNG/JPG/JPEG/WEBP")

    if accepted:
        print(f"\n  ЗАГРУЖЕНО УСПЕШНО: {len(accepted)} файл(ов)")
        print()
        for name in accepted:
            path = os.path.join(INPUT_DIR, name)
            size_kb = os.path.getsize(path) / 1024
            print(f"    {name} ({size_kb:.0f} КБ)")

            # Превью загруженного изображения
            try:
                display(IPImage(filename=path, width=400))
            except Exception:
                print(f"    (превью недоступно)")
            print()

        print("-" * 60)
        print("\n  СЛЕДУЮЩИЙ ШАГ:")
        print("  1. Запустите ячейку 6 (Запуск ComfyUI)")
        print("  2. В ComfyUI: подключите ноду LoadImage к CogVideo sampler")
        print(f"  3. Выберите файл: {accepted[0]}")
    else:
        print("\n  Ни один файл не прошёл валидацию.")
        print("  Поддерживаемые форматы: PNG, JPG, JPEG, WEBP")

    print()
    all_files = os.listdir(INPUT_DIR)
    print(f"  Все файлы в input/: {all_files}")

In [ ]:
#@title 6. Запуск ComfyUI
#@markdown ---
#@markdown ### Способ подключения
#@markdown **ngrok** — самый стабильный туннель (требует бесплатный токен).
#@markdown Cloudflare — без регистрации. Localtunnel — запасной вариант.
tunnel_method = "ngrok" #@param ["ngrok", "cloudflare", "localtunnel"]
ngrok_token = "" #@param {type:"string"}
#@markdown > Токен ngrok: [dashboard.ngrok.com/get-started/your-authtoken](https://dashboard.ngrok.com/get-started/your-authtoken)

import subprocess, time, re, os, requests, shutil

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
LOG_FILE = "/content/comfyui.log"

# Убиваем предыдущий экземпляр ComfyUI если есть
!kill -9 $(lsof -t -i:8188) 2>/dev/null || true

comfy = subprocess.Popen(
    ["python", "main.py", "--listen", "0.0.0.0", "--port", "8188",
     "--enable-cors-header", "*"],
    cwd="/content/ComfyUI",
    stdout=open(LOG_FILE, "w"),
    stderr=subprocess.STDOUT
)

print("Запуск ComfyUI...")
started = False
for i in range(24):
    time.sleep(5)
    try:
        if requests.get("http://localhost:8188/api/system_stats", timeout=3).status_code == 200:
            print(f"  ComfyUI запущен за {(i+1)*5} сек!")
            started = True
            break
    except Exception:
        pass

if not started:
    print("  ComfyUI не ответил за 120 сек.")
    with open(LOG_FILE) as f:
        lines = f.readlines()
        errors = [l for l in lines if any(w in l.lower() for w in ['error', 'traceback', 'exception', 'failed'])]
        if errors:
            print("  --- Ошибки ---")
            for l in errors[-15:]:
                print(f"  {l.rstrip()}")
        for l in lines[-10:]:
            print(f"  {l.rstrip()}")
    raise RuntimeError("ComfyUI не запустился. Проверьте ошибки выше.")

with open(LOG_FILE) as f:
    log_text = f.read()
import_errors = [l for l in log_text.split('\n') if 'cannot import' in l.lower() or 'no module named' in l.lower()]
if import_errors:
    print(f"\n  Предупреждения при загрузке нод ({len(import_errors)}):")
    for l in import_errors[:5]:
        print(f"    {l.strip()}")

url = None
if tunnel_method == "ngrok":
    !pip install pyngrok -q
    from pyngrok import ngrok
    if not ngrok_token:
        import getpass
        ngrok_token = getpass.getpass("Введите ngrok authtoken (dashboard.ngrok.com → Your Authtoken): ")
    ngrok.set_auth_token(ngrok_token)
    tunnel = ngrok.connect(8188)
    url = tunnel.public_url

elif tunnel_method == "cloudflare":
    if not shutil.which("cloudflared"):
        !wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb && dpkg -i cloudflared-linux-amd64.deb 2>&1 | tail -1
    tunnel = subprocess.Popen(
        ["cloudflared", "tunnel", "--url", "http://localhost:8188"],
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT
    )
    time.sleep(5)
    for _ in range(20):
        line = tunnel.stdout.readline().decode()
        match = re.search(r"https://[\w-]+\.trycloudflare\.com", line)
        if match:
            url = match.group(0)
            break

elif tunnel_method == "localtunnel":
    !npm install -g localtunnel > /dev/null 2>&1
    tunnel = subprocess.Popen(
        ["lt", "--port", "8188"],
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT
    )
    time.sleep(8)
    for _ in range(20):
        line = tunnel.stdout.readline().decode()
        match = re.search(r"https://[\w-]+\.loca\.lt", line)
        if match:
            url = match.group(0)
            break

if url:
    print(f"\n{'='*60}")
    print(f"  ComfyUI готов! Откройте: {url}")
    print(f"{'='*60}")
    if tunnel_method == "ngrok":
        print(f"\n  ngrok туннель стабилен — не требует обновления страницы.")
    if tunnel_method == "localtunnel":
        print(f"\n  Localtunnel попросит пароль — нажмите ссылку на странице.")
    print(f"\n  Загрузите воркфлоу: Меню -> Load -> video_cogvideo_t2v")
    print(f"\n  Если страница белая:")
    print(f"    1. Обновите страницу (Ctrl+R)")
    print(f"    2. Если не помогло — перезапустите с другим tunnel_method")
    print(f"\n  Логи ComfyUI: !cat {LOG_FILE}")
else:
    print(f"\nURL туннеля не найден. Попробуйте другой tunnel_method (ngrok / cloudflare / localtunnel).")

In [ ]:
#@title 7. Сохранение на Google Drive
from google.colab import drive
import shutil, glob, os

drive.mount('/content/drive')

src = "/content/ComfyUI/output"
dst = "/content/drive/MyDrive/ComfyUI_Output/cogvideo"
os.makedirs(dst, exist_ok=True)

videos = glob.glob(f"{src}/*.mp4") + glob.glob(f"{src}/*.png")
if not videos:
    print("Результатов пока нет. Сначала сгенерируйте видео!")
else:
    for v in videos:
        shutil.copy2(v, dst)
        print(f"Скопировано: {os.path.basename(v)}")
    print(f"\n{len(videos)} файлов сохранено в {dst}")

---
## Руководство по использованию

### Шаг 1: Откройте ComfyUI
Скопируйте ссылку из вывода ячейки 6 (`https://xxx.trycloudflare.com`) и откройте в браузере.

### Шаг 2: Загрузите воркфлоу
**Меню** -> **Load** -> выберите `video_cogvideo_t2v`

### Шаг 3: Настройте промпт
Отредактируйте текст в ноде промпта. См. советы ниже.

### Шаг 4: Запустите генерацию
Нажмите **Queue Prompt** и дождитесь результата.

---

### Режим T2V (Текст-в-Видео)

Самый простой режим — просто введите описание и нажмите Queue:

| Параметр | Значение по умолчанию | Примечание |
|----------|----------------------|------------|
| Разрешение | 720x480 | Не менять на T4 |
| Кадры | 49 | ~6 сек при 8fps |
| Steps | 25-30 | Больше = качественнее, но дольше |
| CFG | 6.0 | Сила следования промпту |

> **Важно:** Не увеличивайте количество кадров выше 49 на T4 — это приведёт к ошибке OOM (нехватка памяти).

---

### Режим I2V (Изображение-в-Видео)

Для анимации изображения выполните дополнительные шаги:

1. **Загрузите изображение** в ячейке 5 (до запуска ComfyUI)
2. **Откройте ComfyUI** и загрузите воркфлоу
3. **Добавьте ноду LoadImage:**
   - Правый клик -> Add Node -> Image -> LoadImage
   - Выберите ваш загруженный файл
4. **Подключите** выход LoadImage ко входу `image` ноды CogVideo sampler
5. **Напишите промпт**, описывающий желаемое движение
6. Нажмите **Queue Prompt**

> Подсказка: для I2V промпт должен описывать **движение**, а не сцену — модель уже «видит» изображение.

---

### Советы по промптам

CogVideoX лучше всего работает с **описательными, конкретными промптами на английском языке**.

**Описания сцен** (T2V):
> "A serene lake at sunset, mountains in the background, gentle ripples on the water surface, golden hour lighting, cinematic"

**Экшн и движение** (T2V / I2V):
> "A cat jumping gracefully from a shelf, slow motion, soft natural lighting, shallow depth of field"

**Абстракции и стилизация** (T2V):
> "Flowing liquid gold forming into geometric shapes, dark background, reflective surface, satisfying loop"

**I2V — описание движения:**
> "The woman slowly turns her head and smiles, wind blowing through her hair, gentle camera push-in"

| Работает хорошо | Работает плохо |
|-----------------|----------------|
| Конкретные детали движения | Абстрактные команды ("сделай красиво") |
| Описание освещения и атмосферы | Слишком длинные промпты (>100 слов) |
| Один объект / одна сцена | Много объектов одновременно |
| Медленное, плавное движение | Быстрые, резкие движения |

---

### Сравнение с Wan 2.2

| Аспект | CogVideoX-5B | Wan 2.2 14B |
|--------|-------------|-------------|
| Движение | Более плавное, «текучее» | Чёткое, реалистичное |
| Эстетика | Кинематограф, стилизация | Фотореализм |
| Разрешение | 720x480 | до 832x480 |
| Время генерации | Быстрее (~2-3 мин) | Дольше (~5-10 мин) |
| Режимы | T2V, I2V | T2V, I2V, V2V, Talking |

**Рекомендация:** используйте обе модели для разнообразия контента. CogVideoX хорош для стилизованных клипов, Wan — для реалистичных сцен.